# Freeze Risk Prediction from Daily Weather Station Data

**Goal:** predict whether a freeze (minimum temperature below 1°C) will happen
within the next 10 days, using lagged daily weather readings as features.

The 10 input files follow the standard ECA&D station variable codes:
`cc` cloud cover, `hu` humidity, `pp` sea-level pressure, `qq` global radiation,
`rr` precipitation, `sd` snow depth, `ss` sunshine duration, `tg` mean temperature,
`tn` minimum temperature, `tx` maximum temperature.

This notebook was reorganized for readability: dead/commented-out experiments were
removed, duplicate cells were merged, and explanatory notes were added throughout.
Logic was **not** changed except where noted (a couple of small, clearly-flagged fixes).

## 1. Imports & config

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from functools import reduce

In [2]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

## 2. Load the raw station files

Each file is one weather variable, all sharing a `date` column.

In [3]:
df_cc = pd.read_csv(config['data']['clean']['file1'], quotechar='"', sep=";")
df_hu = pd.read_csv(config['data']['clean']['file2'], quotechar='"', sep=";")
df_pp = pd.read_csv(config['data']['clean']['file3'], quotechar='"', sep=";")
df_qq = pd.read_csv(config['data']['clean']['file4'], quotechar='"', sep=";")
df_rr = pd.read_csv(config['data']['clean']['file5'], quotechar='"', sep=";")
df_sd = pd.read_csv(config['data']['clean']['file6'], quotechar='"', sep=";")
df_ss = pd.read_csv(config['data']['clean']['file7'], quotechar='"', sep=";")
df_tg = pd.read_csv(config['data']['clean']['file8'], quotechar='"', sep=";")
df_tn = pd.read_csv(config['data']['clean']['file9'], quotechar='"', sep=";")
df_tx = pd.read_csv(config['data']['clean']['file10'], quotechar='"', sep=";")

Merge all 10 variables into one daily table on `date`.

In [4]:
all_data = reduce(
    lambda left, right: pd.merge(left, right, on=['date'], how='outer'),
    [df_cc, df_hu, df_pp, df_qq, df_rr, df_sd, df_ss, df_tg, df_tn, df_tx]
)
all_data.date = pd.to_datetime(all_data.date, format='%Y-%m-%d')

## 3. Quick exploration on the raw merged data

In [5]:
# plt.figure(figsize=(9, 7))
# sns.heatmap(all_data.corr(), annot=True, fmt=".2f", cmap="coolwarm")
# plt.title("Correlation between raw weather variables")
# plt.show()

In [6]:
all_data.sd.value_counts()

sd
0.0     17013
1.0        49
2.0        26
4.0        17
3.0        16
5.0         7
6.0         5
8.0         4
7.0         4
11.0        3
10.0        2
12.0        2
9.0         1
22.0        1
20.0        1
16.0        1
18.0        1
15.0        1
13.0        1
Name: count, dtype: int64

In [7]:
all_data["year"] = all_data["date"].dt.year

# how many days per year actually had snow on the ground?
all_data.groupby('year')['sd'].apply(lambda x: (x > 0).sum())
real_temps= all_data['tn']

Decided against creating a model to predict snow, pivoted to icy/very cold temps.

## 4. Feature engineering

In [8]:
all_data['tn'] = all_data.tn < 1
all_data["year"] = all_data["date"].dt.year
all_data["day_of_year"] = all_data["date"].dt.dayofyear

# Cyclical encoding of day-of-year, so Dec 31 and Jan 1 are treated as
# neighbours instead of being 364 days apart.
all_data["doy_sin"] = np.sin(2 * np.pi * all_data["day_of_year"] / 365.25)
all_data["doy_cos"] = np.cos(2 * np.pi * all_data["day_of_year"] / 365.25)

# Lag features: only use *past* values to predict the future, never same-day
# or future values, which would leak information the model wouldn't actually
# have at prediction time.
all_data["tn_lag1"] = all_data["tn"].shift(1)
all_data["tn_lag3"] = all_data["tn"].shift(3)
all_data["tn_lag7"] = all_data["tn"].shift(7)

all_data["sd_lag1"] = all_data["sd"].shift(1)
all_data["sd_lag3"] = all_data["sd"].shift(3)
all_data["sd_lag7"] = all_data["sd"].shift(7)
all_data["sd_lag10"] = all_data["sd"].shift(10)

all_data["tx_lag1"] = all_data["tx"].shift(1)
all_data["tg_lag1"] = all_data["tg"].shift(1)

all_data["pp_lag1"] = all_data["pp"].shift(1)
all_data["hu_lag1"] = all_data["hu"].shift(1)
all_data["rr_lag1"] = all_data["rr"].shift(1)
all_data["cc_lag1"] = all_data["cc"].shift(1)

# Drop the first few rows where lag features are undefined (NaN).
all_data = all_data.dropna()

In [9]:
# Post-engineering correlation check, now that lag/cyclical features exist.
all_data.corr()

,date,cc,hu,pp,qq,rr,sd,ss,tg,tn,...,sd_lag1,sd_lag3,sd_lag7,sd_lag10,tx_lag1,tg_lag1,pp_lag1,hu_lag1,rr_lag1,cc_lag1
date,1.000000,-0.098507,-0.085620,-0.009827,-0.164930,0.010738,-0.048220,-0.006393,0.109126,-0.070175,...,-0.048208,-0.048183,-0.048936,-0.052260,0.102989,0.109459,-0.009684,-0.085898,0.010771,-0.098439
cc,-0.098507,1.000000,0.513794,-0.247959,-0.436846,0.236671,-0.003745,-0.756708,-0.128214,-0.086293,...,-0.008985,-0.002526,0.009089,0.023314,-0.234462,-0.167889,-0.115717,0.310073,0.153135,0.450352
hu,-0.085620,0.513794,1.000000,-0.234638,-0.671991,0.287788,0.047568,-0.733442,-0.457638,0.176215,...,0.043693,0.044610,0.053617,0.057935,-0.522401,-0.456599,-0.166153,0.686235,0.228104,0.303943
pp,-0.009827,-0.247959,-0.234638,1.000000,0.135667,-0.350394,-0.021368,0.227692,0.009187,0.106613,...,-0.017513,-0.007513,0.000230,-0.000329,0.056930,0.004975,0.821694,-0.197975,-0.396651,-0.214220
qq,-0.164930,-0.436846,-0.671991,0.135667,1.000000,-0.151648,-0.049717,0.777140,0.561339,-0.228776,...,-0.047515,-0.048119,-0.047015,-0.049764,0.616666,0.551787,0.089225,-0.506355,-0.099514,-0.218309
rr,0.010738,0.236671,0.287788,-0.350394,-0.151648,1.000000,-0.000272,-0.235349,-0.016714,-0.041037,...,-0.001679,-0.008824,0.003195,-0.012129,-0.036753,-0.007548,-0.232102,0.099821,0.173229,0.084393
sd,-0.048220,-0.003745,0.047568,-0.021368,-0.049717,-0.000272,1.000000,-0.031527,-0.138666,0.161750,...,0.777360,0.402361,0.076473,0.005969,-0.130060,-0.138917,-0.019887,0.042483,0.015136,0.000220
ss,-0.006393,-0.756708,-0.733442,0.227692,0.777140,-0.235349,-0.031527,1.000000,0.400724,-0.096026,...,-0.027848,-0.031760,-0.030494,-0.037687,0.496584,0.401543,0.126930,-0.449271,-0.131299,-0.321818
tg,0.109126,-0.128214,-0.457638,0.009187,0.561339,-0.016714,-0.138666,0.400724,1.000000,-0.570093,...,-0.130616,-0.114477,-0.091044,-0.087059,0.964890,0.952681,0.050850,-0.442359,-0.024295,-0.126041
tn,-0.070175,-0.086293,0.176215,0.106613,-0.228776,-0.041037,0.161750,-0.096026,-0.570093,1.000000,...,0.156258,0.135753,0.093394,0.078799,-0.501235,-0.548007,0.072102,0.136088,-0.076860,-0.115515


## 5. Build the modeling table and the target

Select just the features that will actually go into the model, then define
the target: **will a freeze (`tn < 1`) happen at any point in the next 10
days?** This is what `shift(-10)` does -- it looks 10 days *forward*.

In [10]:
df_ice = all_data.copy()

features = [
    'tn_lag1', 'tn_lag3', 'tn_lag7',
    'tg_lag1', 'tx_lag1',
    'cc', 'hu', 'pp', 'rr',
    'doy_sin', 'doy_cos'
]

df_ice = df_ice[features + ['tn', 'year', 'date']]

In [11]:
df_ice['freeze_10d'] = (df_ice['tn'].shift(-10) < 1)

# The last 10 rows now have no future data to look at, so they're NaN --
# drop them rather than feeding the model a meaningless label.
df_ice = df_ice.dropna(subset=['freeze_10d'])

## 6. Train/test split -- chronological, not random

Splitting by year (train on ≤2021, test on >2021) instead of a random split
keeps the time order intact and avoids leaking future information into
training, which a random split would do on time-series data like this.

In [12]:
train = df_ice[df_ice.year <= 2021]
test  = df_ice[df_ice.year > 2021]

y_train = train['freeze_10d']
y_test  = test['freeze_10d']

X_train = train[features]
X_test  = test[features]

**Class balance check** -- how rare is a 'freeze in the next 10 days' event, in the raw data and in each split?

In [13]:
print("Overall freeze-day rate (tn):")
print(all_data['tn'].value_counts(normalize=True))

print("\ny_train class counts:")
print(y_train.value_counts())

print("\ny_test class counts:")
print(y_test.value_counts())

Overall freeze-day rate (tn):
tn
False    0.875241
True     0.124759
Name: proportion, dtype: float64

y_train class counts:
freeze_10d
True     13679
False     2006
Name: count, dtype: int64

y_test class counts:
freeze_10d
True     1326
False     134
Name: count, dtype: int64


## 7. Scaling

In [14]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

print(f"Train shape: {X_train.shape}")

normalizer = MinMaxScaler()
normalizer.fit(X_train)
X_train_norm = pd.DataFrame(normalizer.transform(X_train), columns=X_train.columns)
X_test_norm  = pd.DataFrame(normalizer.transform(X_test),  columns=X_test.columns)

Train shape: (15685, 11)


In [15]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report

# forest = RandomForestClassifier(n_estimators=100, max_depth=20)
# forest.fit(X_train_norm, y_train)
# y_pred_test_rf = forest.predict(X_test_norm)

# print(f"Accuracy: {forest.score(X_test_norm, y_test):.2f}")

In [16]:
# print(classification_report(y_test, y_pred_test_rf))

In [17]:
# pd.Series(
#     forest.feature_importances_,
#     index=X_train.columns
# ).sort_values(ascending=False)

## 8. 10 day model: class-balanced Random Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = all_data.copy()
df = df[features + ['tn', 'year', 'date']]

df['y'] = (df['tn'].shift(-10) < 1)
df = df.dropna(subset=['y'])

train = df[df.year <= 2021]
test  = df[df.year > 2021]

y_train = train['y']
y_test  = test['y']
X_train = train[features]
X_test  = test[features]

model_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    class_weight="balanced"
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

       False       0.28      0.74      0.41       134
        True       0.97      0.81      0.88      1326

    accuracy                           0.80      1460
   macro avg       0.63      0.77      0.65      1460
weighted avg       0.91      0.80      0.84      1460



In [23]:
print(f"Accuracy: {model_balanced.score(X_test, y_test):.2f}")

Accuracy: 0.80


In [24]:
pd.Series(
    model_balanced.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

doy_cos    0.312859
tx_lag1    0.191610
tg_lag1    0.155673
doy_sin    0.090722
pp         0.071624
hu         0.065439
rr         0.037652
cc         0.031601
tn_lag1    0.015220
tn_lag7    0.014555
tn_lag3    0.013046
dtype: float64

## 8. 5 day model: class-balanced Random Forest

In [20]:
df = all_data.copy()
df = df[features + ['tn', 'year', 'date']]

df['y'] = (df['tn'].shift(-5) < 1)
df = df.dropna(subset=['y'])

train = df[df.year <= 2021]
test  = df[df.year > 2021]

y_train = train['y']
y_test  = test['y']
X_train = train[features]
X_test  = test[features]

model_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    class_weight="balanced"
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

       False       0.30      0.67      0.41       132
        True       0.96      0.84      0.90      1328

    accuracy                           0.83      1460
   macro avg       0.63      0.76      0.66      1460
weighted avg       0.90      0.83      0.86      1460



In [21]:
print(f"Accuracy: {model_balanced.score(X_test, y_test):.2f}")

Accuracy: 0.83


## Summary

Two Random Forest classifiers predict whether a freeze will occur within the
next 10 days, using lagged temperature/humidity/pressure/precipitation
features plus a cyclical day-of-year encoding, split chronologically into a
pre-2022 training set and a post-2021 test set:

- **Baseline** (`forest`): 100 trees, max depth 20, trained on min-max scaled
  features.
- **Final** (`model_balanced`): 200 trees, max depth 15, `class_weight="balanced"`,
  trained on unscaled features, with a fixed random seed for reproducibility.